# LLMPreSS · Interactive semantic search

Find passages in **ClimbMix** that are semantically relevant to your question—even when they use different words.

Run the cells in order, then use the search box to explore the corpus. The default edition is **compact**. An index and encoder download is required on first use; the size check below runs before loading the index. Colab storage and memory vary by runtime, and downloads disappear when a temporary runtime is reset.

[Project on GitHub](https://github.com/my-unknown-account/LLMPreSS) · [Index collection on Hugging Face](https://huggingface.co/collections/JamshidJDMY/llmpress)

CPU works by default. For GPU query encoding, select **Runtime → Change runtime type → GPU** before running the cells. Vector search uses the CPU.

### Install

Install Transformers 5 and Hugging Face Hub 1.x together to avoid the Hub downgrade caused by Transformers 4. If you already ran the older installation cell, use **Runtime → Restart session** after this installation, then continue with the index setup cells.

In [ ]:
%pip install -q "git+https://github.com/my-unknown-account/LLMPreSS.git" "transformers>=5,<6" "huggingface_hub>=1.32.0,<2" ipywidgets

# After upgrading an existing runtime, restart the session before continuing.
from importlib.metadata import version
for package in ("llmpress", "transformers", "huggingface_hub"):
    print(f"{package}: {version(package)}")

## 1. Choose the index and check disk space

This checks the published index file sizes without downloading the index. Available disk space is only one requirement; loading and searching also require sufficient RAM. Use a larger runtime if this index exceeds Colab's resources.

In [ ]:
from pathlib import Path
import shutil
from huggingface_hub import HfApi

corpus = "ClimbMix"
edition = "compact"
index_dir = Path("/content/llmpress-index") if Path("/content").exists() else Path("llmpress-index")
repo_id = f"JamshidJDMY/LLMPreSS_{corpus}_{edition}"
required_files = {"index.faiss", "index.json", "lists.ivfdata"}
files = list(HfApi().list_repo_tree(repo_id, repo_type="dataset", recursive=True))
selected = [f for f in files if f.path in required_files]
missing = required_files - {f.path for f in selected}
if missing:
    raise ValueError(f"Published index is missing files: {sorted(missing)}")
index_bytes = sum(f.size for f in selected)
free_bytes = shutil.disk_usage(Path.cwd()).free
print(f"Index: {corpus} / {edition}")
print(f"Index files: {index_bytes / 2**30:.2f} GiB")
print(f"Free disk space: {free_bytes / 2**30:.2f} GiB")
print("The encoder and Python dependencies need additional disk space.")
if not index_dir.exists() and index_bytes >= free_bytes:
    raise RuntimeError("Not enough disk space for this index. Select a larger runtime before continuing.")

## 2. Load the index

Run this cell when you are ready to download. It downloads the index if necessary and loads the encoder once. Keep this cell's state to reuse the index for successive searches. The passage text is retrieved from the configured HTTP API when you search.

In [ ]:
import torch
from llmpress import SemanticIndex

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Query encoder device: {device}")
index = SemanticIndex(
    corpus,
    edition=edition,
    index_dir=index_dir,
    verbose=True,
    threads=32,
    device=device,
)
print("Ready. Run the next cell to open the search interface.")

## 3. Search by meaning

Enter a question or describe a topic. Results show the passage text, similarity score, and source metadata when available. Higher scores indicate stronger matches within the search; they are not probabilities. You can run multiple searches without reloading the index.

In [ ]:
import html
import ipywidgets as widgets
from IPython.display import HTML, display

query_box = widgets.Textarea(
    value="How do plants adapt to drought?",
    placeholder="Ask a question or describe a topic…",
    description="Query:",
    layout=widgets.Layout(width="100%", height="90px"),
)
top_k = widgets.IntSlider(value=5, min=1, max=20, description="Results:")
search_button = widgets.Button(description="Search passages", button_style="primary", icon="search")
output = widgets.Output()
results = []
last_query = None

def run_search(_):
    global results, last_query
    query = query_box.value.strip()
    output.clear_output(wait=True)
    with output:
        if not query:
            print("Enter a question or topic first.")
            return
        search_button.disabled = True
        results = []
        try:
            print("Searching…")
            results = index.search(query, top_k=top_k.value)
            last_query = query
            output.clear_output(wait=True)
            if not results:
                print("No passages found.")
            for result in results:
                title = html.escape(str(result.passage_id or result.numeric_id))
                text = html.escape(result.text or "Passage text unavailable.")
                source = " · ".join(
                    f"{key}: {value}" for key, value in (
                        ("Corpus", result.corpus),
                        ("Source", result.source_rel_path),
                        ("Group", result.source_group),
                        ("Record", result.source_record_index),
                    ) if value is not None
                )
                display(HTML(
                    f'<div style="border:1px solid #94a3b8;border-radius:12px;padding:18px;margin:12px 0">'
                    f'<strong>{result.rank}. {title}</strong> · Score: {result.score:.4f}'
                    f'<p style="white-space:pre-wrap;line-height:1.6">{text}</p>'
                    f'<small>{html.escape(source)}</small></div>'
                ))
        except Exception as error:
            print(f"Search failed: {error}")
        finally:
            search_button.disabled = False

search_button.on_click(run_search)
display(widgets.VBox([query_box, top_k, search_button, output]))

## 4. Export the latest results (optional)

After a successful search, run this cell to save the query and results as JSON. In Colab, the browser will download the file.

In [ ]:
import json
from pathlib import Path

if not results:
    raise ValueError("Run a successful search before exporting.")
export_path = Path("llmpress-results.json")
payload = {
    "query": last_query,
    "results": [result.to_dict() for result in results],
}
export_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
try:
    from google.colab import files
except ImportError:
    print(f"Saved to {export_path.resolve()}")
else:
    files.download(str(export_path))